# 🚀 Workshop Day 1: Data Engineering — Turning Junk into Fuel
## Agentic RAG: From Zero to Hero

**Duration:** 3 hours | **Level:** Undergraduate

---

### 🎯 Goals for Today

We will learn the end-to-end **Data Engineering Pipeline** for RAG:

```
📄 Raw documents → 🔍 Check duplicates → ✂️ Chunking → 📝 Markdown → 🔢 Embedding → 🗄️ VectorDB → 🔎 Retrieval
```

> **RAG (Retrieval-Augmented Generation)** is a technique that helps LLMs answer questions more accurately
> by first "retrieving" relevant information before "generating" the answer


> **🇹🇭 About the Thai Language in This Workshop**
>
> This workshop uses **real Thai text data** because it teaches how to build RAG systems for the Thai language. You don't need to read Thai — English translations are provided as inline comments (`# "translation"`).
>
> **Key differences from English that affect RAG:**
> - **No spaces between words** — Thai uses continuous script (`ฉันรักการเรียนรู้` = 5 words, but no spaces). This means standard tokenizers fail; we use [PyThaiNLP](https://pythainlp.github.io/) instead.
> - **No capitalization** — Thai has no uppercase/lowercase distinction, which affects text normalization.
> - **Complex script** — Thai characters include consonants, vowels (above, below, before, and after consonants), and tone marks, making OCR and text processing more challenging.
> - **BM25 needs special handling** — Without proper Thai tokenization, BM25 sparse search treats entire sentences as single tokens, making keyword matching ineffective.
> - **Multilingual embeddings are essential** — We use `intfloat/multilingual-e5-large` which handles Thai well, unlike English-only models.

### 📖 Thai Quick Reference

Common Thai words you'll see in the output and data:

| Thai | Pronunciation | Meaning |
|:-----|:-------------|:--------|
| สวัสดี | sa-wat-dee | Hello |
| ครับ / ค่ะ | krap / ka | Polite particle (male / female) |
| คำถาม | kham-tham | Question |
| คำตอบ | kham-tob | Answer |
| ค้นหา | kon-ha | Search |
| ข้อมูล | kor-moon | Data / Information |
| ผลลัพธ์ | phon-lap | Result |
| ตัวอย่าง | tua-yang | Example |
| ปัญญาประดิษฐ์ | pan-ya pra-dit | Artificial Intelligence |
| การเรียนรู้ | kan rian-roo | Learning |
| เครื่อง | krueang | Machine |
| แพทย์ | phaet | Doctor / Medical |
| ธนาคาร | ta-na-kan | Bank |
| การเกษตร | kan ka-set | Agriculture |
| มหาวิทยาลัย | ma-ha wit-ta-ya-lai | University |

### 🗺️ Pipeline Overview for Today

| Step | Section | Task |
|:----:|---------|------|
| 📁 | Input | Raw Documents (PDF, TXT, DOCX) |
| ⬇️ | 1.1 Deduplication | Check duplicate files using MD5 Hash |
| ⬇️ | 1.2 Chunking | Split text into smaller parts |
| ⬇️ | 1.3-1.4 Markdown | Convert PDF → Markdown (Gemini / Docling) |
| ⬇️ | 1.5 Embedding | Convert text → Vector (multilingual-e5-large) |
| ⬇️ | 1.6 Hybrid Search | Combine Dense + Sparse (BM25) |
| ⬇️ | 1.7 VectorDB | Set up Qdrant |
| ⬇️ | 1.8 Indexing | Upsert vectors into Qdrant |
| 🔎 | 1.9 Retrieval | Search for vectors similar to the query |

> 💡 Today we will do **every step** from data preparation to retrieval — the full foundation of RAG!

### 💡 Why does Thai benefit more from Dense search?

When comparing alpha values, you'll notice **Thai performs better with higher alpha (more Dense)**:

| Factor | English | Thai |
|--------|---------|------|
| BM25 quality | High — spaces provide clean tokens | Low — tokenization errors cause keyword misses |
| Dense embedding quality | High | High (with multilingual model) |
| Recommended alpha | `0.5` (balanced) | `0.7+` (favor Dense) |

**Rule of thumb for Thai RAG:** Start with `alpha=0.7` (70% Dense, 30% Sparse) and tune from there.
Dense embeddings capture *semantic meaning* regardless of tokenization quality, making them more reliable for Thai text.

---
## 📦 Section 0: Install Dependencies

Install all required libraries for today’s workshop

In [16]:
%%time
# Install all libraries
!pip install -q google-genai \
    'docling>=2.31' \
    'docling-ibm-models>=3.4' \
    sentence-transformers \
    qdrant-client \
    langchain-text-splitters \
    rank_bm25 \
    pymupdf \
    pythainlp \
    scikit-learn \
    rich

# Clear docling-models cache to prevent ONNX file mismatch
import shutil, os
cache_path = os.path.expanduser('~/.cache/huggingface/hub/models--ds4sd--docling-models')
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print('🗑️ Cleared docling-models cache')

print('✅ Installation completed!')
print('⚠️ Please restart runtime: Runtime → Restart session → then run all cells again')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.4/397.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.1 MB/s eta 0:00:00
   ━━━━━━

In [17]:
%%time
# Import core libraries
import hashlib
import os
import json
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

print('✅ Import successful!')

✅ Import successful!
CPU times: user 0 ns, sys: 995 µs, total: 995 µs
Wall time: 1.89 ms


### 📂 Prepare Sample Data

We will create Thai-language sample data to use throughout the workshop.

---

#### 📖 What the sample data contains (English summary)

The sample data consists of **5 Thai-language case studies** about AI adoption in Thailand.
Each file contains real-world scenarios that demonstrate how RAG systems work with Thai text.

**💡 The text is intentionally in Thai — this is the data your RAG pipeline will process!**
English translations are provided as `# [EN]` comment blocks inside the code cell below.

| File | Topic | Key Points |
|:-----|:------|:-----------|
| `case_kmitl.txt` | 🎓 **AI at KMITL University** | Smart Campus energy savings (25%), Thai NLP for student feedback, RAG-based AI Tutor for 200+ courses |
| `case_healthcare.txt` | 🏥 **AI in Healthcare** | Siriraj Hospital X-ray analysis (AI 95% vs radiologist 92%), NLP for medical records, Transfer Learning from English models |
| `case_banking.txt` | 🏦 **AI in Banking** | Kasikorn Bank's LLM+RAG chatbot, reduced Call Center workload by 40%, Vector DB + Hybrid Search |
| `case_education.txt` | 📚 **AI in Education** | University Q&A system using RAG, full pipeline: PDF → Chunk → Embed → Search → Answer, 60% instructor workload reduction |
| `case_agriculture.txt` | 🌾 **AI in Agriculture** | Rice disease detection via CNN (50K+ images, 8 diseases), satellite imagery analysis, crop price prediction from news & social media |

In [18]:
%%time
# Create folders for storing data
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

# Sample data: Case study summaries about AI in Thailand
sample_texts = {

    'case_kmitl.txt': '''Case Study: AI at King Mongkut's Institute of Technology Ladkrabang (KMITL)


# [EN] AI at King Mongkut's Institute of Technology Ladkrabang (KMITL):
# KMITL's Faculty of Engineering developed an AI-powered Smart Campus system
# using IoT sensors combined with Machine Learning to analyze energy usage in classrooms.
# Testing showed 25% electricity savings and improved classroom utilization.
# 
# Additionally, the Information Engineering department developed a Thai NLP system
# for analyzing student feedback from evaluation forms using Sentiment Analysis
# and Topic Modeling to categorize issues and improve curricula more effectively.
# 
# Recently, KMITL built an AI Tutor using RAG that answers course questions
# from over 200 subjects' teaching materials. Students can ask questions 24/7,
# with the system searching answers from lecture notes, slides, and textbooks.

คณะวิศวกรรมศาสตร์ KMITL พัฒนาระบบ AI สำหรับ Smart Campus
ใช้ IoT sensor รวมกับ Machine Learning วิเคราะห์การใช้พลังงานในอาคารเรียน
ผลการทดสอบพบว่าลดค่าไฟฟ้าได้ 25% และเพิ่มประสิทธิภาพการใช้ห้องเรียน

นอกจากนี้ สาขาวิศวกรรมสารสนเทศ ยังพัฒนาระบบ NLP ภาษาไทย
สำหรับวิเคราะห์ความคิดเห็นนักศึกษาจากแบบประเมิน
ใช้ Sentiment Analysis และ Topic Modeling จัดกลุ่มปัญหา
ช่วยปรับปรุงหลักสูตรได้ตรงจุดมากขึ้น

ล่าสุด KMITL นำ RAG มาสร้างระบบ AI Tutor
ช่วยตอบคำถามวิชาเรียนจากเอกสารประกอบการสอนกว่า 200 วิชา
นักศึกษาสามารถถามคำถามได้ตลอด 24 ชั่วโมง
ระบบค้นหาคำตอบจาก lecture notes, slides, และตำราเรียน
''',

    'case_healthcare.txt': '''Case Study: AI in Thai Healthcare


# [EN] AI in Thai Healthcare:
# Siriraj Hospital (Thailand's oldest and largest hospital) uses AI for
# Medical Imaging analysis — detecting lung cancer from X-ray images using Deep Learning.
# Test results show AI achieves 95% accuracy vs. radiologists at 92%.
# 
# They also use NLP to analyze Electronic Medical Records (EMR),
# helping doctors summarize patient histories and recommend treatments,
# reducing record reading time from 15 minutes to 2 minutes per case.
# 
# Challenge: Thai medical data is limited, requiring Transfer Learning
# from English language models with additional fine-tuning on Thai data.

โรงพยาบาลศิริราชได้นำ AI มาใช้ในการวิเคราะห์ภาพถ่ายทางการแพทย์ (Medical Imaging)
เช่น การตรวจจับมะเร็งปอดจากภาพ X-ray ด้วย Deep Learning
ผลการทดสอบพบว่า AI มีความแม่นยำ 95% เทียบกับรังสีแพทย์ที่ 92%

นอกจากนี้ยังมีการใช้ NLP วิเคราะห์เวชระเบียนอิเล็กทรอนิกส์ (EMR)
เพื่อช่วยแพทย์สรุปประวัติผู้ป่วยและแนะนำการรักษาที่เหมาะสม
ลดเวลาการอ่านเวชระเบียนจาก 15 นาทีเหลือ 2 นาทีต่อเคส

ความท้าทาย: ข้อมูลทางการแพทย์ภาษาไทยมีจำนวนจำกัด
ต้องใช้เทคนิค Transfer Learning จากโมเดลภาษาอังกฤษ
along with additional fine-tuning on Thai data.''',

    'case_banking.txt': '''Case Study: AI in Banking and Finance


# [EN] AI in Banking and Finance:
# Kasikorn Bank (KBank) developed a Chatbot called KBTG using LLM + RAG
# to answer customer questions about financial products.
# 
# The RAG system searches an internal knowledge base containing product documents,
# service terms, and FAQs. The LLM then generates natural answers from retrieved info.
# 
# Results: 40% reduction in Call Center workload, 25% increase in customer satisfaction,
# 24/7 service without waiting for staff.
# 
# Technology: Vector Database for storing embeddings, Hybrid Search (Dense + Sparse).

ธนาคารกสิกรไทยได้พัฒนาระบบ Chatbot ชื่อ KBTG
ที่ใช้ Large Language Model (LLM) ร่วมกับ RAG
ในการตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ทำงานโดยการค้นหาข้อมูลจากฐานความรู้ภายใน
ซึ่งประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
จากนั้น LLM จะสร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center ได้ 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
สามารถให้บริการ 24 ชั่วโมง โดยไม่ต้องรอพนักงาน

เทคโนโลยีที่ใช้: Vector Database สำหรับเก็บ Embedding
and Hybrid Search combining Dense + Sparse to retrieve information relevant to the question.''',

    'case_banking_duplicate.txt': '''Case Study: AI in Banking and Finance

ธนาคารกสิกรไทยได้พัฒนาระบบ Chatbot ชื่อ KBTG
ที่ใช้ Large Language Model (LLM) ร่วมกับ RAG
ในการตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ทำงานโดยการค้นหาข้อมูลจากฐานความรู้ภายใน
ซึ่งประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
จากนั้น LLM จะสร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center ได้ 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
สามารถให้บริการ 24 ชั่วโมง โดยไม่ต้องรอพนักงาน

เทคโนโลยีที่ใช้: Vector Database สำหรับเก็บ Embedding
and Hybrid Search combining Dense + Sparse to retrieve information relevant to the question.''',

    'case_education.txt': '''Case Study: AI in Thai Education


# [EN] AI in Thai Education:
# Multiple Thai universities are adopting AI for teaching,
# such as Intelligent Tutoring Systems that adapt content to student levels.
# 
# A notable example uses RAG to build automated Q&A systems for courses,
# embedding lecture materials, slides, and textbooks into Vector Database.
# 
# When students ask questions, the system retrieves relevant content
# and uses LLM to generate answers with source references.
# 
# Key steps: (1) PDF to Markdown, (2) Chunking, (3) Thai-compatible Embedding,
# (4) Upsert to Qdrant, (5) Hybrid Search, (6) LLM answer generation.
# 
# Result: Students can self-study 24/7, reducing instructor Q&A workload by 60%+.

มหาวิทยาลัยหลายแห่งในไทยเริ่มนำ AI มาช่วยในการเรียนการสอน
เช่น ระบบ Intelligent Tutoring System ที่ปรับเนื้อหาตามระดับของผู้เรียน

ตัวอย่างที่น่าสนใจคือการใช้ RAG สร้างระบบถาม-ตอบอัตโนมัติ
สำหรับวิชาเรียน โดยนำเอกสารประกอบการสอน Slides และหนังสือ
มา Embed เป็น Vector แล้วเก็บใน Vector Database

เมื่อนักศึกษาถามคำถาม ระบบจะค้นหาเนื้อหาที่เกี่ยวข้อง
แล้วใช้ LLM สร้างคำตอบพร้อมอ้างอิงแหล่งที่มา

ขั้นตอนสำคัญในการสร้างระบบนี้:
1. เตรียมข้อมูล: แปลง PDF เป็น Markdown
2. ตัดข้อความ: ใช้ Chunking แบ่งเนื้อหาเป็นส่วนย่อย
3. สร้าง Embedding: แปลง Chunk เป็น Vector ด้วยโมเดลที่รองรับภาษาไทย
4. จัดเก็บ: Upsert Vector เข้า Qdrant หรือ Vector DB อื่น
5. ค้นหา: ใช้ Hybrid Search ค้นหา Chunk ที่เกี่ยวข้อง
6. สร้างคำตอบ: ส่ง Context ให้ LLM สร้างคำตอบ

ผลลัพธ์: นักศึกษาสามารถเรียนรู้ด้วยตนเองได้ตลอด 24 ชั่วโมง
reducing repeated question-answering workload for instructors by more than 60%.''',

    'case_agriculture.txt': '''Case Study: AI in Smart Agriculture


# [EN] AI in Smart Agriculture:
# Smart Farming in Thailand uses AI to analyze satellite imagery
# and IoT sensor data for yield prediction and crop disease surveillance.
# 
# Computer Vision detects rice diseases from leaf photos using CNN
# trained on 50,000+ images covering 8 major rice diseases.
# 
# NLP analyzes agricultural commodity prices from news, government reports,
# and social media to help farmers make planting and selling decisions.
# 
# Challenges: Data scattered across agencies, specialized Thai agricultural terminology,
# and Data Engineering needed to consolidate and clean data first.

Smart Farming ในประเทศไทยใช้ AI วิเคราะห์ภาพถ่ายดาวเทียม
และข้อมูลจาก IoT Sensor เพื่อพยากรณ์ผลผลิตและเฝ้าระวังโรคพืช

ระบบ Computer Vision สามารถตรวจจับโรคข้าวได้จากภาพถ่ายใบข้าว
โดยใช้ Convolutional Neural Network (CNN) ที่ Train ด้วยภาพถ่าย
โรคข้าวกว่า 50,000 ภาพ ครอบคลุม 8 โรคหลัก

นอกจากนี้ยังมีการใช้ NLP วิเคราะห์ข้อมูลราคาสินค้าเกษตร
จากข่าวสาร รายงานภาครัฐ และ Social Media
เพื่อช่วยเกษตรกรตัดสินใจเรื่องการปลูกและการขาย

ความท้าทายของ AI ในการเกษตรไทย:
- ข้อมูลกระจัดกระจายอยู่ในหลายหน่วยงาน
- ภาษาและศัพท์เฉพาะทางการเกษตรไทย
- Data Engineering is required to consolidate and clean data first'''
}

for fname, content in sample_texts.items():
    with open(f'data/{fname}', 'w', encoding='utf-8') as f:
        f.write(content)

print(f'✅ Created {len(sample_texts)} sample files in the data/ folder')
print()
for fname in sorted(sample_texts.keys()):
    print(f'  📄 {fname}')

✅ Created 5 sample files in the data/ folder

  📄 case_agriculture.txt
  📄 case_banking.txt
  📄 case_banking_duplicate.txt
  📄 case_education.txt
  📄 case_healthcare.txt
CPU times: user 12 µs, sys: 999 µs, total: 1.01 ms
Wall time: 2.05 ms


---
## 🔍 Section 1.1: Check Duplicates with MD5

### What is MD5 Hash?

**MD5 (Message Digest Algorithm 5)** is an algorithm that converts any data into a 128-bit hash value (32 hex characters)

- Same data → always produces the same hash
- Even slightly different data → produces a completely different hash

### Why check for duplicates?

In RAG work, if duplicate documents exist:
- ❌ Wastes storage and embedding time
- ❌ Retrieval results become repetitive
- ❌ The LLM may get confused by duplicated context

In [19]:
%%time
def compute_md5(filepath):
    """Compute the MD5 hash of a file"""
    md5_hash = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b''):
            md5_hash.update(chunk)
    return md5_hash.hexdigest()

# Test: Compute MD5 for all files
print('📊 MD5 Hash of each file:')
print('=' * 70)
file_hashes = {}
for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath):
        h = compute_md5(fpath)
        file_hashes[fname] = h
        print(f'  {fname:<25} → {h}')

print('\n🔍 Notice: case_banking.txt and case_banking_duplicate.txt have the same hash!')

📊 MD5 Hash of each file:
  case_agriculture.txt      → 1d03b92b2dbb2d89eeb0c053207fe283
  case_banking.txt          → 66160d60bd9880b6b155915e9f0b7c1d
  case_banking_duplicate.txt → 66160d60bd9880b6b155915e9f0b7c1d
  case_education.txt        → 8e47b92d1afdb2f61800890133ffe7f7
  case_healthcare.txt       → 2c56ffd8024fe12efc351c587515339d

🔍 Notice: case_banking.txt and case_banking_duplicate.txt have the same hash!
CPU times: user 0 ns, sys: 942 µs, total: 942 µs
Wall time: 1.76 ms


In [20]:
%%time
def find_duplicates(directory):
    """Find duplicate files in a folder using MD5"""
    hash_map = {}  # md5 → [list of filepaths]

    for fname in os.listdir(directory):
        fpath = os.path.join(directory, fname)
        if os.path.isfile(fpath):
            h = compute_md5(fpath)
            hash_map.setdefault(h, []).append(fname)

    # Filter only groups with more than 1 file
    duplicates = {h: files for h, files in hash_map.items() if len(files) > 1}
    return duplicates, hash_map

duplicates, hash_map = find_duplicates('data')

if duplicates:
    print('⚠️ Duplicate files found!')
    for h, files in duplicates.items():
        print(f'\n  Hash: {h}')
        print(f'  Duplicate files: {", ".join(files)}')
        print(f'  → Keep: {files[0]}, remove: {", ".join(files[1:])}')
else:
    print('✅ No duplicate files found')

⚠️ Duplicate files found!

  Hash: 66160d60bd9880b6b155915e9f0b7c1d
  Duplicate files: case_banking_duplicate.txt, case_banking.txt
  → Keep: case_banking_duplicate.txt, remove: case_banking.txt
CPU times: user 1 ms, sys: 0 ns, total: 1 ms
Wall time: 7.06 ms


In [21]:
%%time
# Remove duplicate files (keep only the first one)
removed = []
for h, files in duplicates.items():
    for f in files[1:]:  # ลบทุกตัวยกเว้นตัวแรก
        os.remove(f'data/{f}')
        removed.append(f)

print(f'🗑️ Removed {len(removed)} duplicate file(s): {", ".join(removed)}')
print(f'📁 Remaining files: {sorted(os.listdir("data"))}')

🗑️ Removed 1 duplicate file: case_banking.txt
📁 Remaining files: ['case_agriculture.txt', 'case_banking_duplicate.txt', 'case_education.txt', 'case_healthcare.txt']
CPU times: user 0 ns, sys: 900 µs, total: 900 µs
Wall time: 2.41 ms


### 💡 Observations
- `case_banking.txt` and `case_banking_duplicate.txt` produced the **same hash** → content is 100% identical
- Even if filenames are different, MD5 checks the **content**, not the filename
- Removing duplicates → reduces storage + prevents repetitive retrieval results

> 🎯 In real work, there may be thousands of duplicate files — MD5 helps catch them instantly

### 🧪 Exercise 1.1

Try creating 2–3 more files (with some duplicates), then use the `find_duplicates()` function to check them

In [ ]:
%%time
# 🧪 Exercise: Create more files and test duplicate detection

# 💡 Hint:
#   1. Create a new file with open('data/my_file.txt', 'w')
#   2. Try making files with identical content
#   3. Call find_duplicates('data') and inspect the result

# with open('data/test1.txt', 'w') as f:
#     f.write('test content')
# with open('data/test2.txt', 'w') as f:
#     f.write('test content')  # duplicate!
# find_duplicates('data')

---
## ✂️ Section 1.2: Different Types of Chunking

### Why do we need Chunking?

- LLMs have a **limited context window** — you cannot send an entire 100-page document
- Embedding models work best with **short to medium-length text** (no more than 512 tokens)
- Proper chunks help **improve retrieval accuracy**

### Chunking methods we will learn

| Method | Principle | Pros | Cons |
|------|---------|-------|--------|
| Fixed-size | Split by character count | Simple, fast | May cut in the middle of a sentence |
| Recursive | Split using nested separators | Better than fixed | Must define separators |
| Sentence-based | Split by sentence | Preserves meaning | Chunk sizes are inconsistent |
| Semantic | Group using embeddings | Most accurate | Slow, expensive |

In [22]:
%%time
# Load sample text
all_docs = {}  # เก็บแต่ละเอกสารแยกกัน
all_text = ''

for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath) and fname.endswith('.txt'):
        with open(fpath, 'r', encoding='utf-8') as f:
            content = f.read()
            all_docs[fname] = content
            all_text += '\n\n' + content

print(f'📄 Number of documents: {len(all_docs)} files')
print(f'📄 Total text length: {len(all_text)} characters')
for fname, content in all_docs.items():
    print(f'  📄 {fname}: {len(content)} characters')

📄 Number of documents: 4 files
📄 Total text length: 2628 characters
  📄 case_agriculture.txt: 629 characters
  📄 case_banking_duplicate.txt: 567 characters
  📄 case_education.txt: 877 characters
  📄 case_healthcare.txt: 547 characters
CPU times: user 859 µs, sys: 21 µs, total: 880 µs
Wall time: 5.96 ms


### Method 1: Fixed-size Chunking

In [23]:
%%time
def fixed_size_chunk(text, chunk_size=200, overlap=50):
    """Split text by a fixed character count"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # เลื่อนกลับมาซ้อนเพื่อไม่ให้ขาดบริบท
    return chunks

fixed_chunks = fixed_size_chunk(all_text, chunk_size=200, overlap=50)
print(f'📊 Fixed-size: got {len(fixed_chunks)} chunks')
print(f'\n--- Chunk 1 ---')
print(fixed_chunks[0])
print(f'\n--- Chunk 2 ---')
print(fixed_chunks[1])

📊 Fixed-size: got 18 chunks

--- Chunk 1 ---


Case Study: AI in Smart Agriculture

Smart Farming in Thailand uses AI to analyze satellite images
and data from IoT sensors to forecast yields and monitor crop diseases.

Computer Vision systems can detect rice disea

--- Chunk 2 ---
ses.

Computer Vision systems can detect rice diseases from rice leaf images
using Convolutional Neural Networks (CNN) trained on
more than 50,000 rice disease images covering 8 major diseases.

In addition, NLP is used to ana
CPU times: user 205 µs, sys: 18 µs, total: 223 µs
Wall time: 231 µs


### Method 2: Recursive Character Chunking (LangChain)

This method will try to **split using the first separator first**. If the chunk is still too large → it tries the next separator

```python
separators = ['\n\n', '\n', '。', r'\. ', ' ', '']
#              ①      ②    ③     ④    ⑤   ⑥
```

| Order | Separator | Meaning | Example |
|:-----:|-----------|----------|----------|
| ① | `\n\n` | **Blank line** (new paragraph) | Split between paragraphs |
| ② | `\n` | **New line** | Split line by line |
| ③ | `。` | **Chinese/Japanese period** | Sentence boundary in CJK |
| ④ | `\. ` | **English period + space** | "Hello. World" → split at the period |
| ⑤ | ` ` | **Space** | Split word by word |
| ⑥ | `''` | **Every character** (last resort) | Split in the middle of a word |

**How it works:**
```
Very long text → first try splitting by \n\n
                  ↓ if chunk is still too long
                  try splitting by \n
                  ↓ if still too long
                  try splitting by periods
                  ↓ ...
                  split by individual characters (last resort)
```

> 💡 **For Thai**: there are no spaces between words, so the separator ` ` is usually not very useful
> The main separators will be `\n\n` and `\n`

In [24]:
%%time
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=['\n\n', '\n', '。', r'\. ', ' ', ''],  # ลำดับการแบ่ง
)

recursive_chunks = recursive_splitter.split_text(all_text)
print(f'📊 Recursive: got {len(recursive_chunks)} chunks')
print(f'\n--- Chunk 1 ---')
print(recursive_chunks[0])
print(f'\n--- Chunk 2 ---')
print(recursive_chunks[1])

📊 Recursive: got 17 chunks

--- Chunk 1 ---
Case Study: AI in Smart Agriculture

Smart Farming in Thailand uses AI to analyze satellite images
and data from IoT sensors to forecast yields and monitor crop diseases.

--- Chunk 2 ---
Computer Vision systems can detect rice diseases from rice leaf images
using Convolutional Neural Networks (CNN) trained on
more than 50,000 images covering 8 major rice diseases.
CPU times: user 27.3 s, sys: 2.93 s, total: 30.2 s
Wall time: 55.6 s


### Method 3: Sentence-based Chunking

In [10]:
%%time
import re

def sentence_chunk(text, max_sentences=3):
    """Split text by sentence/line (suitable for Thai)"""
    # Thai mainly uses newline as a sentence boundary here
    sentences = re.split(r'\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = '\n'.join(sentences[i:i+max_sentences])
        if chunk:
            chunks.append(chunk)
    return chunks

sentence_chunks = sentence_chunk(all_text, max_sentences=3)
print(f'📊 Sentence-based: got {len(sentence_chunks)} chunks')

# Show the first 3 chunks
for idx in range(min(3, len(sentence_chunks))):
    print(f'\n--- Chunk {idx+1} ({len(sentence_chunks[idx])} characters) ---')
    print(sentence_chunks[idx])

📊 Sentence-based: got 18 chunks

--- Chunk 1 (150 characters) ---
Case Study: AI in Smart Agriculture
Smart Farming in Thailand uses AI to analyze satellite images
and data from IoT sensors to forecast yields and monitor crop diseases.

--- Chunk 2 (166 characters) ---
Computer Vision systems can detect rice diseases from rice leaf images
using Convolutional Neural Networks (CNN) trained on
more than 50,000 images covering 8 major diseases.

--- Chunk 3 (143 characters) ---
In addition, NLP is used to analyze agricultural commodity price data
from news, government reports, and social media
to help farmers make decisions about planting and selling.
CPU times: user 761 µs, sys: 0 ns, total: 761 µs
Wall time: 769 µs


### 📊 Compare the Results

In [25]:
%%time
print('📊 Compare number of chunks:')
print(f'  Fixed-size (200 chars):     {len(fixed_chunks)} chunks')
print(f'  Recursive (200 chars):      {len(recursive_chunks)} chunks')
print(f'  Sentence-based (3 sent):    {len(sentence_chunks)} chunks')

print('\n📏 Average chunk size:')
for name, chunks in [('Fixed', fixed_chunks), ('Recursive', recursive_chunks), ('Sentence', sentence_chunks)]:
    sizes = [len(c) for c in chunks]
    print(f'  {name:<12}: avg={np.mean(sizes):.0f}, min={min(sizes)}, max={max(sizes)} characters')

📊 Compare number of chunks:
  Fixed-size (200 chars):     18 chunks
  Recursive (200 chars):      17 chunks
  Sentence-based (3 sent):    18 chunks

📏 Average chunk size:
  Fixed       : avg=193, min=78, max=200 characters
  Recursive   : avg=156, min=98, max=198 characters
  Sentence    : avg=144, min=40, max=187 characters
CPU times: user 304 µs, sys: 3 µs, total: 307 µs
Wall time: 297 µs


### 💡 Observations
- **Fixed-size** cuts in the middle of sentences → information is lost, context is damaged
- **Recursive** tries to split by paragraph/sentence → better than Fixed
- **Sentence-based** preserves complete sentences, but chunk sizes are inconsistent
- **overlap** helps prevent information loss at boundaries

> 🎯 For Thai: **Recursive** is the best choice in terms of quality/speed

### 🧪 Exercise 1.2

Try adjusting `chunk_size` and `overlap`, then compare the results. Which settings are best for Thai text?

In [ ]:
%%time
# 🧪 Exercise: Try adjusting chunk_size and overlap

# 💡 Hint:
#   1. chunk_size=100 vs 500 → how many chunks do you get?
#   2. overlap=0 vs 100 → is any text lost at the boundaries?

# for size in [100, 200, 500]:
#     chunks = fixed_size_chunk(all_text, chunk_size=size, overlap=50)
#     print(f'chunk_size={size}: {len(chunks)} chunks')

---
## 🤖 Section 1.3: Convert Documents to Markdown with Gemini

### What is Gemini Multimodal?

**Multimodal** = understands text + images + PDF + video

```
Upload PDF → Gemini can "see" the page
    ↓
Understands layout: header, table, bullet points, images
    ↓
Converts it into structured Markdown
```

### Pros vs Cons

| | Pros | Cons |
|---|------|--------|
| ✅ | Understands complex layout | Requires internet |
| ✅ | Supports Thai | Uses API credits |
| ✅ | Handles tables/images | Data is sent to the cloud |

> ⚠️ **Sensitive documents** (patient data, contracts) → consider using Docling instead (Section 1.4)

In [12]:
%%time
# Configure Gemini API
# 🔑 You can create an API Key at: https://aistudio.google.com/apikey
#
# How to set up Colab Secrets:
#   1. Click the 🔑 icon in the left sidebar
#   2. Click 'Add new secret'
#   3. Set the name to: GOOGLE_API_KEY
#   4. Paste the API Key you copied
#   5. Turn on the 'Notebook access' toggle

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('✅ Loaded API Key from Colab Secrets successfully!')
except Exception:
    GOOGLE_API_KEY = input('🔑 กรุณาวาง Gemini API Key ของคุณ: ')  # "🔑 Please paste your Gemini API Key: "

from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)

# Test that the API works
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='ตอบว่า สวัสดี เท่านั้น'  # "Reply only with Hello"
)
print(f'✅ Gemini API test successful: {response.text.strip()}')

✅ Loaded API Key from Colab Secrets successfully!
✅ Gemini API test successful: Hello
CPU times: user 7.83 s, sys: 446 ms, total: 8.28 s
Wall time: 28 s


### Create a Sample PDF for Testing

We will create a sample PDF from Thai text

In [26]:
%%time
import fitz  # PyMuPDF
import subprocess

# Install Thai font on Colab
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-thai-tlwg'],
               capture_output=True)
print('✅ Thai font installed successfully')

# Find the path to the Thai font
import glob
thai_fonts = glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf')
THAI_FONT = thai_fonts[0] if thai_fonts else None
print(f'📝 Using font: {THAI_FONT}')

def create_sample_pdf(filepath, text, title='เอกสารตัวอย่าง'):  # "Sample Document"
    """Create a PDF from Thai text"""
    doc = fitz.open()
    page = doc.new_page()

    # Load Thai font
    thai_font = fitz.Font(fontfile=THAI_FONT)

    # Write title
    tw = fitz.TextWriter(page.rect)
    y = 50
    tw.append((50, y), title, font=thai_font, fontsize=18)
    y += 40

    # Write content line by line
    for line in text.split('\n'):
        if y > 750:  # ขึ้นหน้าใหม่
            tw.write_text(page)
            page = doc.new_page()
            tw = fitz.TextWriter(page.rect)
            y = 50
        tw.append((50, y), line, font=thai_font, fontsize=11)
        y += 18

    tw.write_text(page)
    doc.save(filepath)
    doc.close()

# Read text and create a PDF
with open('data/case_healthcare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

create_sample_pdf('data/sample.pdf', text, 'กรณีศึกษา: AI ในการแพทย์ไทย')  # "data/sample.pdf"
print('✅ Created sample.pdf successfully (supports Thai!)')

✅ Thai font installed successfully
📝 Using font: /usr/share/fonts/truetype/tlwg/Kinnari-BoldItalic.ttf
✅ Created sample.pdf successfully (supports Thai!)
CPU times: user 127 ms, sys: 17.1 ms, total: 144 ms
Wall time: 13.1 s


In [27]:
%%time
# Use Gemini to convert PDF to Markdown
try:
    # Use Gemini to convert PDF to Markdown
    import pathlib

    def pdf_to_markdown_gemini(pdf_path, client):
        """Use Gemini to read a PDF and convert it to Mark..."""
        # Read PDF file
        pdf_bytes = pathlib.Path(pdf_path).read_bytes()

        prompt = '''Read this PDF document and convert all of its content into Markdown format
    กฎ:
    - รักษาโครงสร้างเดิม (หัวข้อ, ย่อหน้า, ตาราง)
    - ใช้ heading (#, ##, ###) ตามลำดับความสำคัญ
    - ถ้ามีตาราง ให้แปลงเป็น Markdown table
    - ถ้ามีรายการ ให้ใช้ bullet points
    - Keep the content complete; do not summarize or omit anything'''

        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                genai.types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
                prompt
            ]
        )
        return response.text

    # Test
    markdown_output = pdf_to_markdown_gemini('data/sample.pdf', client)
    print('📝 Markdown output from Gemini:')
    print('=' * 60)
    display(Markdown(markdown_output))
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Please check: 1) API Key is correct  2) Internet connection works  3) PDF file is valid')


📝 Markdown output from Gemini:


# Case Study: AI in Thai Healthcare

## Case Study: AI in Thai Healthcare

Siriraj Hospital has applied AI to medical image analysis (Medical Imaging), such as detecting lung cancer from X-ray images using Deep Learning. Test results showed that AI achieved 95% accuracy, compared to radiologists at 92%.

In addition, NLP is used to analyze electronic medical records (EMR) to help doctors summarize patient history and recommend appropriate treatment. This reduces chart review time from 15 minutes to 2 minutes per case.

Challenge: Thai medical data is limited in quantity. Transfer Learning from English models is required, along with additional fine-tuning on Thai data.

CPU times: user 15.2 ms, sys: 2.01 ms, total: 17.2 ms
Wall time: 4.31 s


In [28]:
%%time
# Save output
with open('output/gemini_output.md', 'w', encoding='utf-8') as f:
    f.write(markdown_output)
print('✅ Saved output to output/gemini_output.md')

✅ Saved output to output/gemini_output.md
CPU times: user 1.14 ms, sys: 38 µs, total: 1.18 ms
Wall time: 1.27 ms


### 💡 Observations
- Gemini **understands page layout** → it can convert headers, tables, and bullets
- Uses **Multimodal** capabilities → reads PDF as if it can "see" the page
- Downside: requires internet + API cost

> ⚠️ For documents containing sensitive data (contracts, patient data) — consider using Docling instead (Section 1.4)

### 🧪 Exercise 1.3

Try uploading your own PDF (for example, slides from a course) and use Gemini to convert it into Markdown

In [ ]:
%%time
# 🧪 Exercise: Convert your own PDF
# TODO: Upload your PDF and use pdf_to_markdown_gemini()



---
## 📄 Section 1.4: Convert Documents to Markdown with Docling

### What is Docling?

**[Docling](https://github.com/DS4SD/docling)** is an open-source library from IBM
for converting documents (PDF, DOCX, PPTX, HTML) into Markdown or JSON

| | Gemini | Docling |
|---|--------|--------|
| **Type** | Cloud API (LLM) | Local library |
| **Cost** | Limited quota | Free, unlimited |
| **Speed** | Depends on network | Depends on CPU/GPU |
| **Advantage** | Better contextual understanding | Can work offline |
| **Disadvantage** | Requires internet | May not understand complex context as well |

In [29]:
%%time
# Suppress noisy warnings from Docling's OCR backend
import warnings
import logging
warnings.filterwarnings('ignore', category=UserWarning, module='huggingface_hub')
logging.getLogger('RapidOCR').setLevel(logging.WARNING)
logging.getLogger('docling').setLevel(logging.WARNING)

from docling.document_converter import DocumentConverter

# Convert PDF with Docling
converter = DocumentConverter()
result = converter.convert('data/sample.pdf')

docling_markdown = result.document.export_to_markdown()

print('📝 Markdown output from Docling:')
print('=' * 60)
display(Markdown(docling_markdown))

[INFO] 2026-03-06 03:55:08,055 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-06 03:55:08,061 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-06 03:55:08,066 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.7.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:08,951 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-03-06 03:55:09,114 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:09,117 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:09,382 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-06 03:55:09,383 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-06 03:55:09,385 [RapidOCR] download_file.py:68: Initiating download: https://

📝 Markdown output from Docling:


## Case Study: AI in Thai Healthcare

Case Study: AI in Thai Healthcare

Siriraj Hospital has applied AI to medical image analysis (Medical Imaging), such as detecting lung cancer from X-ray images using Deep Learning. Test results showed that AI achieved 95% accuracy, compared to radiologists at 92%.

In addition, NLP is used to analyze electronic medical records (EMR) to help doctors summarize patient history and recommend appropriate treatment. This reduces chart review time from 15 minutes to 2 minutes per case.

Challenge: Thai medical data is limited in quantity. Transfer Learning from English models is required, along with additional fine-tuning on Thai data.

CPU times: user 13 s, sys: 6.69 s, total: 19.7 s
Wall time: 35.3 s


### ⚠️ Note: Docling OCR and Thai text

Docling uses **RapidOCR** as its default OCR engine, which downloads **Chinese PP-OCR models** (~40 MB).
These models are designed for Chinese characters and **may not accurately recognize Thai text** in scanned PDFs.

**For this workshop**, our sample PDF is **text-based** (not scanned), so text extraction works fine without OCR.
For scanned Thai documents in production, consider:
- Using `pytesseract` with the Thai language pack (`tha`)
- Or using Google Cloud Vision API with Thai support

In [30]:
%%time
# Save comparison output
with open('output/docling_output.md', 'w', encoding='utf-8') as f:
    f.write(docling_markdown)

print('📊 Compare Gemini vs Docling:')
print(f'  Gemini output:  {len(markdown_output)} characters')
print(f'  Docling output: {len(docling_markdown)} characters')
print('\n💡 Tip: In real projects, choose based on the situation')
print('  - Complex documents (charts, tables) → Gemini')
print('  - Large numbers of documents, need offline processing → Docling')

📊 Compare Gemini vs Docling:
  Gemini output:  581 characters
  Docling output: 579 characters

💡 Tip: In real projects, choose based on the situation
  - Complex documents (charts, tables) → Gemini
  - Large numbers of documents, need offline processing → Docling
CPU times: user 1.29 ms, sys: 0 ns, total: 1.29 ms
Wall time: 989 µs


### 📊 Compare Gemini vs Docling

| Criteria | Gemini API | Docling (IBM) |
|-------|-----------|---------------|
| **Speed** | ⚡ Fast (API call) | 🐢 Slower (runs locally) |
| **Markdown quality** | ⭐⭐⭐ Very good | ⭐⭐ Good |
| **Thai support** | ✅ Excellent | ⚠️ Usable |
| **Price** | 💰 Free (with quota) | 🆓 Completely free |
| **Requires Internet** | ✅ Yes | ❌ No |
| **Supports tables/images** | ✅ Good | ✅ Very good |
| **Privacy** | ⚠️ Sent to the cloud | ✅ Processed locally |
| **Best for** | Prototypes, general documents | Production, sensitive documents |

### 🧪 Exercise 1.4

Try converting the same PDF with both Gemini and Docling, then compare which one gives better results

In [ ]:
%%time
# 🧪 Exercise: Compare Gemini vs Docling with your own PDF
# TODO: Write your code here



---
## 🔢 Section 1.5: Standard Embedding (Thai-capable)

### What is Embedding?

**Embedding = converting text into numbers (vectors)** that a computer can understand

```
"AI helps doctors"     → [0.23, -0.11, 0.45, ...] (1024 numbers)
"artificial intelligence" → [0.21, -0.13, 0.42, ...] ← close! (similar meaning)
"delicious food"       → [0.89,  0.56, -0.32, ...] ← far apart (different topic)
```

### Why multilingual?

| Model | Thai | Vector Size | Speed |
|-------|:-------:|:-----------:|:--------:|
| `all-MiniLM-L6-v2` | ❌ Poor | 384 | ⚡ Fast |
| `multilingual-e5-large` | ✅ Good | 1024 | 🐢 Slower |
| `paraphrase-multilingual` | ✅ Okay | 768 | ⚡ Medium |

> 💡 We chose **multilingual-e5-large** because it supports Thai best
> (In real use, you may consider domain-specific fine-tuning)

In [31]:
%%time
# Load Embedding Model (the first run will download it from HuggingFace)
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

try:
    embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
    test = embed_model.encode('ทดสอบ')  # "test"
    print(f'✅ Loaded Embedding Model successfully!')
    print(f'   Model: intfloat/multilingual-e5-large')
    print(f'   Vector size: {len(test)}')
except Exception as e:
    print(f'❌ Failed to load model: {e}')
    print('💡 Check: 1) Internet 2) Disk space (requires ~2GB)')

✅ Loaded Embedding Model successfully!
   Model: intfloat/multilingual-e5-large
   Vector size: 1024
CPU times: user 12.5 s, sys: 16.2 s, total: 28.7 s
Wall time: 38.4 s


In [33]:
%%time
# Test Thai embeddings
thai_sentences = [
    'query: ปัญญาประดิษฐ์คืออะไร',  # "query: What is artificial intelligence?"
    'passage: AI คือสาขาของวิทยาการคอมพิวเตอร์ที่สร้างระบบอัจฉริยะ',  # "passage: AI is a branch of computer science tha..."
    'passage: Machine Learning เป็นส่วนหนึ่งของ AI ที่เรียนรู้จากข้อมูล',  # "passage: Machine Learning is part of AI that le..."
    'passage: วันนี้อากาศดีมาก ท้องฟ้าแจ่มใส',  # "passage: The weather is very nice today and the..."
    'passage: Vector Database ใช้เก็บข้อมูลแบบเวกเตอร์',  # "passage: Vector Databases are used to store vec..."
]

# Create embeddings
embeddings = embed_model.encode(thai_sentences)

print(f'📊 Created embeddings successfully!')
print(f'  Number of texts: {len(embeddings)}')
print(f'  Vector size: {embeddings.shape[1]} dimensions')
print(f'  Example vector (first 5 values): {embeddings[0][:5]}')

📊 Created embeddings successfully!
  Number of texts: 5
  Vector size: 1024 dimensions
  Example vector (first 5 values): [ 0.02122011 -0.01509202  0.01514856 -0.05054271  0.05155794]
CPU times: user 2.29 s, sys: 36.2 ms, total: 2.32 s
Wall time: 1.52 s


### 📐 How to Measure Vector Similarity

Once we convert text into vectors, the next step is to **compare** which vectors are similar
There are 3 common methods:

---

**1️⃣ Cosine Similarity** — measures the "angle" between 2 vectors
```
                    A · B           Sum of (a₁×b₁ + a₂×b₂ + ...)
Cosine(A, B) = ─────────── = ─────────────────────────────────────────
                |A| × |B|       Length of A × Length of B

Range:  -1 (opposite) ← 0 (unrelated) → 1 (very similar) ✅
```

**2️⃣ Dot Product** — multiplies each dimension and sums them up
```
Dot(A, B) = a₁×b₁ + a₂×b₂ + a₃×b₃ + ...

Range:  -∞ ← 0 → +∞ (higher = more similar)
```

**3️⃣ L2 Distance (Euclidean)** — measures the "distance" between two points
```
L2(A, B) = √( (a₁-b₁)² + (a₂-b₂)² + (a₃-b₃)² + ... )

Range:  0 (identical) → +∞ (very different)  ⚠️ lower = more similar
```

---

### 🤔 When should you use which one?

**1️⃣ Cosine Similarity — most common for RAG / NLP**

It only cares about the "direction", not the "magnitude" of the vector

```
Example: comparing articles
  Article A: about AI, 10 pages long   → long vector (larger magnitude)
  Article B: about AI, 1 paragraph long → short vector (smaller magnitude)

  Cosine = 0.95 ✅ → both are "about the same topic" even if lengths differ
  Dot Product = very different ❌ → because vector magnitudes differ
```
👉 **Use for**: finding semantically similar documents, RAG retrieval, semantic search

---

**2️⃣ Dot Product — use when vectors are normalized**

If you normalize vectors so their length = 1 → Dot Product = Cosine (same result, but faster)

```
Example: recommendation system
  User vector (normalized):    [0.5, 0.7, 0.1]  (length = 1)
  Product A vector (normalized): [0.4, 0.8, 0.2]  (length = 1)

  Dot = 0.5×0.4 + 0.7×0.8 + 0.1×0.2 = 0.78 ← fast to compute!
```
👉 **Use for**: high-speed systems, normalized vectors, large-scale production systems

---

**3️⃣ L2 Distance — use when "magnitude" matters**

It measures actual spatial distance, like distance on a map

```
Example: grouping similar images (Image Clustering)
  Cat image A: [0.2, 0.8, 0.3]
  Cat image B: [0.3, 0.7, 0.4]  → L2 = 0.17 (close ✅)
  Car image:   [0.9, 0.1, 0.8]  → L2 = 1.12 (far ❌)
```
👉 **Use for**: image similarity, clustering, anomaly detection

---

> 💡 **Simple summary**: If you are building RAG → use **Cosine Similarity** as the default first choice!

In [ ]:
%%time
# ─── Install Thai font for matplotlib ───
import subprocess, glob
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Install Thai font package (available on Colab)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-thai-tlwg'], capture_output=True)

# Register the fonts directly with matplotlib
for font_path in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fm.fontManager.addfont(font_path)

plt.rcParams['font.family'] = 'Garuda'
print('✅ Thai font (Garuda) is ready to use')

In [34]:
%%time
# === Measure similarity in 3 ways — show as Heatmaps ===
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

labels = ['Q: AI คืออะไร', 'P: AI สาขา CS', 'P: ML ส่วนของ AI', 'P: อากาศดี', 'P: VectorDB']  # "Q: What is AI"
short_labels = ['Q: AI?', 'P: AI-CS', 'P: ML-AI', 'P: อากาศ', 'P: VecDB']  # "Q: AI?"

# Compute all 3 methods
cos_sim = cosine_similarity(embeddings)
dot_prod = np.dot(embeddings, embeddings.T)
l2_dist = euclidean_distances(embeddings)

# --- Heatmap Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

matrices = [
    (cos_sim, '1️⃣ Cosine Similarity\n(ยิ่งสูง = ยิ่งคล้าย)', 'YlOrRd', '.3f'),  # "1️⃣ Cosine Similarity\n(higher = more similar)"
    (dot_prod, '2️⃣ Dot Product\n(ยิ่งสูง = ยิ่งคล้าย)', 'YlOrRd', '.1f'),  # "2️⃣ Dot Product\n(higher = more similar)"
    (l2_dist, '3️⃣ L2 Distance\n(ยิ่งต่ำ = ยิ่งคล้าย)', 'YlOrRd_r', '.3f'),  # "3️⃣ L2 Distance\n(lower = more similar)"
]

for ax, (matrix, title, cmap, fmt) in zip(axes, matrices):
    im = ax.imshow(matrix, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(short_labels)))
    ax.set_yticks(range(len(short_labels)))
    ax.set_xticklabels(short_labels, fontsize=10, rotation=45, ha='right')
    ax.set_yticklabels(short_labels, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

    # Show values in each cell
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = matrix[i][j]
            color = 'white' if val > (matrix.max() + matrix.min()) / 2 else 'black'
            ax.text(j, i, f'{val:{fmt}}', ha='center', va='center', fontsize=9, color=color)

    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('📐 เปรียบเทียบ 3 วิธีวัดความคล้ายคลึงของ Vector', fontsize=14, fontweight='bold', y=1.02)  # "📐 Compare 3 Ways to Measure Vector Similarity"
plt.tight_layout()
plt.show()

print('\n💡 Observations:')
print('  - Cosine: AI-related texts have high similarity (~0.9), weather is lower (~0.7)')
print('  - L2: AI-related texts have smaller distance, weather has larger distance')
print('  - Dot Product shows the same trend as Cosine because the vectors are already normalized')

1️⃣ Cosine Similarity (higher = more similar, max=1.0)
                   Q: What is AI   P: AI field CSP: ML part of AI      P: Nice weather     P: VectorDB
   Q: What is AI        1.000           0.850           0.818           0.722           0.754   
   P: AI field CS        0.850           1.000           0.917  ⭐        0.796           0.827   
P: ML part of AI        0.818           0.917  ⭐        1.000           0.806           0.846   
      P: Nice weather        0.722           0.796           0.806           1.000           0.818   
     P: VectorDB        0.754           0.827           0.846           0.818           1.000   


2️⃣ Dot Product (higher = more similar, no max)
                   Q: What is AI   P: AI field CSP: ML part of AI      P: Nice weather     P: VectorDB
   Q: What is AI             1.0             0.8             0.8             0.7             0.8
   P: AI field CS             0.8             1.0             0.9             0.8             0.8
P: 

### 🎨 Visualization: Similar texts = close together

Reduce dimensions from 1024 → 2 using PCA, then plot to see which vectors are close together

In [ ]:
%%time
from sklearn.decomposition import PCA

# Reduce dimensions from 1024 → 2 using PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

# Plot
plt.figure(figsize=(10, 7))

labels = ['Q: AI คืออะไร', 'P: AI สาขา CS', 'P: ML ส่วนของ AI', 'P: อากาศดี', 'P: VectorDB']  # "Q: What is AI"
colors = ['red', 'blue', 'blue', 'gray', 'green']

for j, (x, y) in enumerate(coords):
    plt.scatter(x, y, c=colors[j], s=200, zorder=5)
    plt.annotate(labels[j], (x, y), textcoords='offset points',
                 xytext=(10, 10), fontsize=11,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# Draw lines from the query to passages
for j in range(1, len(coords)):
    plt.plot([coords[0][0], coords[j][0]], [coords[0][1], coords[j][1]],
             'k--', alpha=0.3)

plt.title('📐 Embedding Space (PCA 2D) — ข้อความคล้ายกัน = อยู่ใกล้กัน', fontsize=14)  # "📐 Embedding Space (PCA 2D) — Similar texts = cl..."
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('💡 Observation: AI-related texts (blue) are close together')
print('   while the weather-related text (gray) is farther away')

### 💡 Observations
- Sentences with **similar meaning** → cosine score **close to 1.0**
- Sentences that are **unrelated** → cosine score **close to 0**
- In the PCA plot: texts on the same topic **stay close together** → this is the core idea behind semantic search
- `multilingual-e5-large` understands Thai because it was trained on more than 100 languages

> 🎯 **Embedding is the core of RAG** — if embeddings are poor, retrieval will be inaccurate

### 🧪 Exercise 1.5

Try writing 5 Thai sentences of your own, then compute and compare their similarities

In [ ]:
%%time
# 🧪 Exercise: Try embeddings with your own Thai sentences
# TODO: Write your code here



---
## 🔀 Section 1.6: Hybrid Embedding (Dense + Sparse)

### What is Dense Embedding?

**Dense Embedding** converts text into a vector where **every position has a value** (no zeros)
using a Neural Network to learn the "meaning" of the text

```
"The cat is sleeping on the sofa" → [0.12, -0.34, 0.56, 0.78, 0.23, -0.11, ...] (1024 values, none are zero)
```

✅ **Strong at**: understanding "meaning" — knows that "dog" and "canine" refer to the same thing
❌ **Weak at**: exact keyword retrieval, such as product code "SKU-12345"

---

### What is Sparse Embedding?

**Sparse Embedding** converts text into a vector where **most positions are 0**
and values only appear at positions corresponding to words that occur, using statistical methods like BM25 or TF-IDF

```
Suppose vocabulary = [cat, dog, sleep, run, sofa, garden, ...] (50,000 words)

"The cat is sleeping on the sofa" → [0.8, 0, 0.5, 0, 0.3, 0, 0, 0, 0, ...] (50,000 values, most are 0)
                                   cat  dog sleep run sofa garden
```

✅ **Strong at**: exact keywords such as proper names, codes, technical terms
❌ **Weak at**: synonyms — it treats "dog" and "canine" as different words

---

### 🔀 Hybrid Search — Why combine them?

| Scenario | Dense | Sparse | Hybrid |
|-----------|-------|--------|--------|
| "What is AI" → retrieve "artificial intelligence" | ✅ Yes | ❌ No | ✅ Yes |
| Find code "SKU-12345" | ❌ No | ✅ Yes | ✅ Yes |
| "How to treat diabetes" | ✅ Yes | ✅ Yes | ✅✅ Better |

**Hybrid formula:**
```
Hybrid Score = α × Dense Score + (1-α) × Sparse Score

α = 1.0 → Dense only
α = 0.5 → balanced mix (recommended)
α = 0.0 → Sparse only
```

### 🇹🇭 Why does Thai need a special tokenizer?

Thai **does not have spaces between words**, unlike English:

```
English:  "I love machine learning"
          → just split on spaces → ["I", "love", "machine", "learning"]  ✅ Easy!

Thai:     "ฉันรักการเรียนรู้ของเครื่อง"
          → split on spaces → ["ฉันรักการเรียนรู้ของเครื่อง"]  ❌ only one token!
          → use pythainlp  → ["ฉัน", "รัก", "การ", "เรียนรู้", "ของ", "เครื่อง"]  ✅
```

**BM25 needs tokens (words) to count frequencies** → if you do not tokenize, it cannot find words correctly!

We use `pythainlp.word_tokenize()`, which combines dictionary-based and ML-based Thai tokenization

In [36]:
%%time
from rank_bm25 import BM25Okapi
import pythainlp

# Prepare sample text
documents = [
    'ปัญญาประดิษฐ์ คือสาขาที่สร้างระบบอัจฉริยะ',  # "Artificial intelligence is a field that builds ..."
    'Machine Learning เป็นส่วนหนึ่งของ AI ที่เรียนรู้จากข้อมูล',  # "Machine Learning is part of AI that learns from..."
    'NLP ช่วยให้คอมพิวเตอร์เข้าใจภาษามนุษย์',  # "NLP helps computers understand human language"
    'Vector Database เก็บข้อมูลในรูปแบบเวกเตอร์',  # "Vector Databases store data in vector form"
    'RAG คือเทคนิครวม LLM กับการค้นหาข้อมูล',  # "RAG is a technique that combines LLMs with info..."
    'Qdrant เป็น open-source vector database',  # "Qdrant is an open-source vector database"
]

# Thai tokenization with PyThaiNLP
tokenized_docs = [pythainlp.word_tokenize(doc) for doc in documents]
print('📝 Tokenization examples:')
for i, (doc, tokens) in enumerate(zip(documents[:2], tokenized_docs[:2])):
    print(f'  [{i}] {doc}')
    print(f'      → {tokens}')

# Create BM25 index (Sparse)
bm25 = BM25Okapi(tokenized_docs)
print(f'\n✅ Created BM25 index successfully ({len(documents)} documents)')

📝 Tokenization examples:
  [0] Artificial intelligence is a field that builds intelligent systems
      → ['Artificial intelligence', ' ', 'is', 'a field', 'that', 'builds', 'intelligent systems']
  [1] Machine Learning is part of AI that learns from data
      → ['Machine', ' ', 'Learning', ' ', 'is', 'part', 'of', ' ', 'AI', ' ', 'that', 'learns', 'from', 'data']

✅ Created BM25 index successfully (6 documents)
CPU times: user 2.82 s, sys: 235 ms, total: 3.05 s
Wall time: 3.15 s


In [37]:
%%time
# Hybrid Search function
def hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5, top_k=3):
    """Hybrid search: combine Dense + Sparse"""
    # Dense search (Semantic)
    q_emb = embed_model.encode([f'query: {query}'])
    doc_embs = embed_model.encode([f'passage: {d}' for d in documents])
    dense_scores = cosine_similarity(q_emb, doc_embs)[0]

    # Sparse search (BM25)
    q_tokens = pythainlp.word_tokenize(query)
    sparse_scores = bm25.get_scores(q_tokens)

    # Normalize scores to [0, 1]
    if max(dense_scores) > 0:
        dense_norm = dense_scores / max(dense_scores)
    else:
        dense_norm = dense_scores

    if max(sparse_scores) > 0:
        sparse_norm = sparse_scores / max(sparse_scores)
    else:
        sparse_norm = sparse_scores

    # Hybrid score
    hybrid_scores = alpha * dense_norm + (1 - alpha) * sparse_norm

    # Rank results
    ranked = sorted(enumerate(zip(documents, dense_norm, sparse_norm, hybrid_scores)),
                    key=lambda x: x[1][3], reverse=True)
    return ranked[:top_k]

# Test
query = 'AI เรียนรู้จากข้อมูล'  # "AI learns from data"
results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5)

print(f'🔍 Query: "{query}"')
print(f'{"":>4}{"Document":<50} {"Dense":>8} {"Sparse":>8} {"Hybrid":>8}')
print('=' * 82)
for idx, (doc, dense, sparse, hybrid) in results:
    print(f'  [{idx}] {doc:<48} {dense:>8.3f} {sparse:>8.3f} {hybrid:>8.3f}')

🔍 Query: "AI learns from data"
    Document                                              Dense   Sparse   Hybrid
  [1] Machine Learning is part of AI that learns from data    1.000    1.000    1.000
  [5] Qdrant is an open-source vector database             0.919    0.138    0.528
  [3] Vector Databases store data in vector form          0.922    0.110    0.516
CPU times: user 2.79 s, sys: 23.6 ms, total: 2.81 s
Wall time: 2.49 s


In [38]:
%%time
# Compare different alpha values
print('📊 Compare different alpha (Dense weight) values:')
print('=' * 70)
query = 'vector database สำหรับ RAG'  # "vector database for RAG"
print(f'Query: "{query}"\n')

for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=alpha, top_k=2)
    top_doc = results[0][1][0]
    top_score = results[0][1][3]
    label = 'Sparse only' if alpha == 0 else 'Dense only' if alpha == 1 else f'Hybrid'
    print(f'  α={alpha:.1f} ({label:<12}): [{results[0][0]}] {top_doc[:45]}... ({top_score:.3f})')

📊 Compare different alpha (Dense weight) values:
Query: "vector database for RAG"

  α=0.0 (Sparse only ): [5] Qdrant is an open-source vector database... (1.000)
  α=0.3 (Hybrid      ): [5] Qdrant is an open-source vector database... (0.994)
  α=0.5 (Hybrid      ): [5] Qdrant is an open-source vector database... (0.991)
  α=0.7 (Hybrid      ): [5] Qdrant is an open-source vector database... (0.987)
  α=1.0 (Dense only  ): [3] Vector Databases store data in vector form... (1.000)
CPU times: user 13.8 s, sys: 228 ms, total: 14 s
Wall time: 14.4 s


### 💡 Observations
- **Dense** (embedding) is strong at **meaning** → "treat patients" = "healthcare"
- **Sparse** (BM25) is strong at **exact words** → "Python" is found more precisely
- **alpha=0.7** (Dense 70% + Sparse 30%) often works best for Thai
- Thai requires a special tokenizer (PyThaiNLP) because there are no spaces between words

> 🎯 **Hybrid = meaning + exact words** → more accurate than using only one approach

### 🧪 Exercise 1.6

Try changing the query to pure Thai and pure English, then see which alpha value gives the best results

In [ ]:
%%time
# 🧪 Exercise: Try Hybrid search with Thai queries

# 💡 Hint:
#   1. Try different alpha values: 0.0 (Sparse only), 0.5 (50/50), 1.0 (Dense only)
#   2. Try keyword-based queries vs semantic queries

# hybrid_search('Deep Learning', all_chunks, embed_model, bm25,
#               tokenized_docs, alpha=0.5, top_k=3)

---
## 🗄️ Section 1.7: Set Up Qdrant VectorDB

### What is a Vector Database?

**Vector Database** = a database designed to store and search vectors quickly

```
Text → Embedding → Vector [0.12, -0.45, 0.78, ...] → store in VectorDB
                                                           ↕
Query   → Embedding → Vector → search for the nearest vectors (ANN)
```

### Why choose Qdrant?

| VectorDB | Language | Cloud | In-Memory | Open Source |
|----------|------|:-----:|:---------:|:-----------:|
| **Qdrant** | Rust | ✅ | ✅ | ✅ |
| ChromaDB | Python | ❌ | ✅ | ✅ |
| Pinecone | — | ✅ | ❌ | ❌ |
| Weaviate | Go | ✅ | ✅ | ✅ |

> 💡 We choose **Qdrant** because it is fast (Rust), supports in-memory mode (great for Colab), and allows filtering with payloads

In [44]:
%%time
from qdrant_client import QdrantClient, models

try:
    qdrant = QdrantClient(':memory:')
    print('✅ Connected to Qdrant successfully! (in-memory mode)')
    print('💡 in-memory = stored in RAM, fast but everything is lost when Colab is closed')
except Exception as e:
    print(f'❌ Could not connect to Qdrant: {e}')

✅ Connected to Qdrant successfully! (in-memory mode)
💡 in-memory = stored in RAM, fast but everything is lost when Colab is closed
CPU times: user 10.1 ms, sys: 0 ns, total: 10.1 ms
Wall time: 12.1 ms


In [47]:
COLLECTION_NAME = 'thai_documents'
VECTOR_SIZE = 1024

In [49]:
%%time
COLLECTION_NAME = 'thai_documents'
VECTOR_SIZE = 1024

# Check whether the Collection already exists; if so, delete it first
if qdrant.collection_exists(collection_name=COLLECTION_NAME):
    qdrant.delete_collection(collection_name=COLLECTION_NAME)
    print(f'🗑️ Deleted existing Collection "{COLLECTION_NAME}"')

# Create Collection
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE,
        distance=models.Distance.COSINE,  # ใช้ Cosine similarity
    ),
)

# Verify
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ Created Collection "{COLLECTION_NAME}" successfully!')
print(f'  Status: {info.status}')
print(f'  Vector size: {info.config.params.vectors.size}')
print(f'  Distance: {info.config.params.vectors.distance}')
print(f'  Points: {info.points_count}')

🗑️ Deleted existing Collection "thai_documents"
✅ Created Collection "thai_documents" successfully!
  Status: green
  Vector size: 1024
  Distance: Cosine
  Points: 0
CPU times: user 999 µs, sys: 0 ns, total: 999 µs
Wall time: 1.01 ms


### 💡 Observations
- `:memory:` = stored in memory → fast, but everything disappears when Colab is closed
- `Cosine` = compares vector "direction" → suitable for NLP
- Real production → use Qdrant Cloud or Docker + persistent storage

> 🎯 VectorDB makes it possible to **search through millions of vectors in milliseconds** using ANN algorithms

### 🧪 Exercise 1.7

Try creating a new collection named `my_collection` with `EUCLID` as the distance metric

In [ ]:
# 🧪 Exercise: Create your own collection
# TODO: Write your code here



---
## 📥 Section 1.8: Build the Index (Upsert Data)

### What is Indexing?

**Indexing** = the process of putting chunk + embedding + metadata into the VectorDB

```
For each chunk:
  1. Embed: convert chunk → vector
  2. Create Point: id + vector + payload (text, source)
  3. Upsert: insert into Qdrant (if id already exists, update it)
```

| Term | Meaning |
|--------|----------|
| **Point** | A unit of data in Qdrant (id + vector + payload) |
| **Payload** | Metadata attached to the vector (text, source, date, ...) |
| **Upsert** | Insert if new / Update if already exists |

In [50]:
%%time
# Prepare data for indexing
# Read all documents → chunk them using recursive splitter

all_chunks = []
for fname in sorted(os.listdir('data')):
    if not fname.endswith('.txt'):
        continue
    with open(f'data/{fname}', 'r', encoding='utf-8') as f:
        text = f.read()

    chunks = recursive_splitter.split_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text': chunk,
            'source': fname,
            'chunk_index': i,
        })

print(f'📊 Data prepared successfully:')
print(f'  Total chunks: {len(all_chunks)}')
for fname in sorted(os.listdir('data')):
    if fname.endswith('.txt'):
        count = sum(1 for c in all_chunks if c['source'] == fname)
        print(f'  {fname}: {count} chunks')

📊 Data prepared successfully:
  Total chunks: 18
  case_agriculture.txt: 4 chunks
  case_banking_duplicate.txt: 4 chunks
  case_education.txt: 6 chunks
  case_healthcare.txt: 4 chunks
CPU times: user 0 ns, sys: 1.51 ms, total: 1.51 ms
Wall time: 1.39 ms


In [42]:
%%time
# Create embeddings for all chunks
print('⏳ Creating embeddings...')
chunk_texts = [f"passage: {c['text']}" for c in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
print(f'✅ Created embeddings successfully! ({len(chunk_embeddings)} vectors × {chunk_embeddings.shape[1]} dimensions)')

⏳ Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Created embeddings successfully! (18 vectors × 1024 dimensions)
CPU times: user 18.7 s, sys: 266 ms, total: 18.9 s
Wall time: 11.5 s


In [51]:
%%time
# Upsert into Qdrant
points = []
for i, (chunk, embedding) in enumerate(zip(all_chunks, chunk_embeddings)):
    points.append(
        models.PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                'text': chunk['text'],
                'source': chunk['source'],
                'chunk_index': chunk['chunk_index'],
            }
        )
    )

# Upsert all at once
qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

# Verify
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ Upsert successful! Number of points in collection: {info.points_count}')

✅ Upsert successful! Number of points in collection: 18
CPU times: user 19.8 ms, sys: 814 µs, total: 20.6 ms
Wall time: 21.1 ms


### 💡 Observations
- **payload** = metadata attached to the vector (such as text, source)
- This helps enable filtering, such as searching only `source='healthcare'`
- **upsert** = insert + update → if the id already exists, it will be updated instead

> 🎯 Good payload design → makes retrieval and filtering more flexible

### 🧪 Exercise 1.8

Try adding new data into the collection (for example, text about a topic you are interested in)

In [ ]:
# 🧪 Exercise: Add your own data
# TODO: Write your code here



---
## 🔎 Section 1.9: Retrieve Data from Qdrant

### What is Retrieval?

**Retrieval** = finding chunks relevant to a query from the VectorDB

```
Query: "How does AI help doctors?"
  ↓ embed
Vector: [0.23, -0.11, ...]
  ↓ search in Qdrant (Cosine Similarity)
Top-3 Results:
  1. [score=0.92] "Siriraj Hospital uses AI to analyze X-ray images..."
  2. [score=0.85] "NLP analyzes electronic medical records..."
  3. [score=0.71] "Deep Learning is used to detect cancer..."
```

### Adjustable settings

| Parameter | Function | Recommended value |
|-----------|--------|----------|
| `top_k` | Number of chunks to retrieve | 3-5 (balance accuracy/coverage) |
| `score_threshold` | Filter out low-score results | 0.5+ |
| `filter` | Filter by metadata | source, date, category |

### Dense Search (Semantic Search)

### 🎯 What is top_k?

**`top_k`** is the number of results you want the system to return, ordered by score from highest to lowest

```
Example: if there are 100 chunks in the database

top_k=3  → get the 3 most similar chunks ✅ fast, concise context
top_k=5  → get the 5 most similar chunks    more context
top_k=10 → get the 10 most similar chunks   ⚠️ may include noise
```

**Choosing the right top_k:**

| top_k | Suitable for | Pros | Cons |
|-------|---------|-------|--------|
| 1-3 | Specific questions | Fast, focused | May miss important information |
| 3-5 | General questions (recommended) | Good balance | — |
| 5-10 | Broad / complex questions | More coverage | Slower, longer context |

> 💡 **In practice**: top_k=3~5 is common in RAG
> because LLMs work better with concise context rather than overly long input

In [52]:
%%time
def search_qdrant(query, top_k=3):
    """Search data from Qdrant using dense embeddings"""
    # Create query embedding
    q_embedding = embed_model.encode([f'query: {query}'])[0]

    # Search
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding.tolist(),
        limit=top_k,
        with_payload=True,
    )
    return results

# Test retrieval
query = 'ปัญญาประดิษฐ์คืออะไร'  # "What is artificial intelligence?"
results = search_qdrant(query)

print(f'🔍 Query: "{query}"')
print(f'📊 Top-3 results:')
print('=' * 70)
for i, point in enumerate(results.points):
    print(f'\n🏆 Rank {i+1} (Score: {point.score:.4f})')
    print(f'   Source: {point.payload["source"]} (chunk {point.payload["chunk_index"]})')
    print(f'   Content: {point.payload["text"][:150]}...')

🔍 Query: "What is artificial intelligence?"
📊 Top-3 results:

🏆 Rank 1 (Score: 0.7913)
   Source: case_agriculture.txt (chunk 0)
   Content: Case Study: AI in Smart Agriculture

Smart Farming in Thailand uses AI to analyze satellite images
and data from IoT sensors to forecast yields and monitor crop disea...

🏆 Rank 2 (Score: 0.7865)
   Source: case_healthcare.txt (chunk 0)
   Content: Case Study: AI in Thai Healthcare...

🏆 Rank 3 (Score: 0.7815)
   Source: case_healthcare.txt (chunk 1)
   Content: Siriraj Hospital has applied AI to medical image analysis (Medical Imaging)
such as detecting lung cancer from X-ray images using Deep Learning
Test results showed th...
CPU times: user 543 ms, sys: 14.6 ms, total: 558 ms
Wall time: 369 ms


### Try Searching Other Questions

In [53]:
%%time
# Test multiple questions
test_queries = [
    'Vector Database คืออะไร',  # "What is a Vector Database?"
    'RAG ทำงานอย่างไร',  # "How does RAG work?"
    'Deep Learning กับ Machine Learning ต่างกันอย่างไร',  # "What is the difference between Deep Learning an..."
]

for query in test_queries:
    results = search_qdrant(query, top_k=2)
    print(f'\n🔍 Query: "{query}"')
    for i, point in enumerate(results.points):
        print(f'   [{i+1}] (Score: {point.score:.4f}) {point.payload["text"][:80]}...')
    print('-' * 70)


🔍 Query: "What is a Vector Database?"
   [1] (Score: 0.8410) Technologies used: A Vector Database for storing embeddings
Hybrid Search combining Dense + ...
   [2] (Score: 0.8151) 4. Store data: Upsert vectors into Qdrant or another Vector DB
5. Search: Use Hybrid S...
----------------------------------------------------------------------

🔍 Query: "How does RAG work?"
   [1] (Score: 0.8712) The RAG system works by retrieving information from an internal knowledge base
which includes product docume...
   [2] (Score: 0.8572) One interesting example is using RAG to build an automated question-answering system
for courses by taking...
----------------------------------------------------------------------

🔍 Query: "What is the difference between Deep Learning and Machine Learning?"
   [1] (Score: 0.8206) Technologies used: A Vector Database for storing embeddings
Hybrid Search combining Dense + ...
   [2] (Score: 0.8100) 4. Store data: Upsert vectors into Qdrant or another Vector DB
5. S

### Search with Metadata Filters

In [54]:
%%time
# Search only within a specific document
query = 'การเรียนรู้ของเครื่อง'  # "machine learning"
q_embedding = embed_model.encode([f'query: {query}'])[0]

results_filtered = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=q_embedding.tolist(),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key='source',
                match=models.MatchValue(value='case_healthcare.txt'),
            )
        ]
    ),
    limit=3,
    with_payload=True,
)

print(f'🔍 Query: "{query}" (filter: source=case_healthcare.txt)')
for i, point in enumerate(results_filtered.points):
    print(f'   [{i+1}] ({point.score:.4f}) [{point.payload["source"]}] {point.payload["text"][:80]}...')

🔍 Query: "machine learning" (filter: source=case_healthcare.txt)
   [1] (0.7978) [case_healthcare.txt] Case Study: AI in Thai Healthcare...
   [2] (0.7856) [case_healthcare.txt] Challenge: Thai medical data is limited in quantity
Transfer Learning is required...
   [3] (0.7817) [case_healthcare.txt] Siriraj Hospital has applied AI to medical image analysis (Medical Imaging)...
CPU times: user 486 ms, sys: 10.2 ms, total: 496 ms
Wall time: 312 ms


### 💡 Observations
- A score close to **1.0** = very similar, close to **0** = unrelated
- The query **does not need to match the exact text** → "AI helps doctors" can still retrieve "disease diagnosis"
- Small `top_k` (3-5) → faster and more accurate | Large `top_k` (10-20) → broader but noisier
- **Filters** help narrow the search scope → faster and more relevant results

> 🎯 **Retrieval is the most important step in RAG** — retrieve wrong = answer wrong

### 🧪 Exercise 1.9

1. Try searching other Thai questions
2. Try using filters to search only from a selected document
3. Try changing `top_k` and observe how the results change

In [ ]:
%%time
# 🧪 Exercise: Try searching from Qdrant

# 💡 Hint:
#   1. Try specific queries vs broad queries
#   2. Change top_k → how do the results change?
#   3. Try filtering with different sources

# search_qdrant('Thai rice', top_k=5)
# search_qdrant('AI', top_k=3)  # too broad?

---
## 🚀 Bonus: End-to-End RAG Demo

Let’s connect everything you learned today and send the retrieved chunks into Gemini to generate an answer!

```
Question → Retrieval (Qdrant) → Top Chunks → LLM (Gemini) → Answer
```

> 💡 This is a preview of **Day 2**, where we will learn how to build a full Agent!

In [56]:
%%time
def rag_answer(question, top_k=3):
    """A simple RAG system: retrieve + generate answer"""
    # 1. Embed query
    query_vector = embed_model.encode(f'query: {question}').tolist()

    # 2. Retrieve from Qdrant
    results = qdrant.query_points(
        collection_name='thai_documents',
        query=query_vector,
        limit=top_k
    ).points

    # 3. Build context from retrieved chunks
    context_parts = []
    print(f'🔍 Question: {question}')
    print(f'📚 Retrieved {len(results)} chunks:')
    for idx, r in enumerate(results):
        text = r.payload['text']
        context_parts.append(text)
        print(f'  [{idx+1}] score={r.score:.4f} | {text[:60]}...')

    context = '\n\n'.join(context_parts)

    # 4. Send to Gemini to generate an answer
    prompt = f'''จากบริบทต่อไปนี้ ตอบคำถามเป็นภาษาไทย กระชับ ชัดเจน
ถ้าไม่มีข้อมูลเพียงพอให้บอกว่า "ข้อมูลไม่เพียงพอ"  # "Insufficient information"

บริบท:
{context}

คำถาม: {question}
Answer:'''

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )

    print(f'\n🤖 Answer from RAG:')
    print(f'   {response.text.strip()}')
    print()
    return response.text.strip()

# Test with multiple questions
questions = [
    'AI ช่วยในวงการแพทย์ไทยอย่างไร',  # "How does AI help in Thai healthcare?"
    'RAG ใช้ในธนาคารอย่างไร',  # "How is RAG used in banking?"
    'ขั้นตอนสร้างระบบ AI ช่วยสอนมีอะไรบ้าง',  # "What are the steps to build an AI teaching assi..."
    'มหาวิทยาลัยใช้ AI อย่างไรได้บ้าง'  # "How can universities use AI?"
]

for q in questions:
    print('=' * 60)
    rag_answer(q)
    print()

🔍 Question: How does AI help in Thai healthcare?
📚 Retrieved 3 chunks:
  [1] score=0.8733 | Siriraj Hospital has applied AI to medical image analysis...
  [2] score=0.8680 | Case Study: AI in Thai Healthcare...
  [3] score=0.8512 | Case Study: AI in Thai Education

Many universities in Thailand have started applying...

🤖 Answer from RAG:
   AI helps analyze medical images, such as detecting lung cancer from X-ray images, with accuracy higher than radiologists.


🔍 Question: How is RAG used in banking?
📚 Retrieved 3 chunks:
  [1] score=0.8503 | Case Study: AI in Banking and Finance

Kasikornbank has developed...
  [2] score=0.8480 | One interesting example is using RAG to build an automated question-answering system
for cou...
  [3] score=0.8456 | The RAG system works by retrieving information from an internal knowledge base
which contains...

🤖 Answer from RAG:
   RAG is used in banking (for example, by Kasikornbank through the KBTG chatbot) to answer customer questions about financia

---
## 🎯 Summary: What We Learned Today

### The complete Data Engineering pipeline for RAG

| Step | Section | Task |
|:----:|---------|------|
| 📄 | Input | Raw Documents (PDF, TXT, DOCX) |
| 🔍 | 1.1 | Check Duplicates (MD5) → remove duplicate files |
| 📝 | 1.3-1.4 | Convert to Markdown (Gemini / Docling) |
| ✂️ | 1.2 | Chunking (Fixed / Recursive / Sentence) |
| 🔢 | 1.5 | Dense Embedding (multilingual-e5-large) |
| 🔀 | 1.6 | Hybrid Embedding (Dense + BM25) |
| 🗄️ | 1.7 | Create Qdrant Collection |
| 📥 | 1.8 | Index (Upsert vectors + metadata) |
| 🔎 | 1.9 | Retrieve (Dense / Filtered search) |


### 🔑 Key Takeaways

1. **Data quality matters most** — "Garbage in, garbage out"
2. **Chunking strategy** has a major impact on RAG quality
3. **Hybrid search** (Dense + Sparse) is usually better than using only one of them
4. **Metadata** helps filter results more accurately

### 📅 Next time: Day 2 — Building Agents: Finding the Needle in the Haystack

We will learn how to build **AI Agents** that use the RAG pipeline you created
for more intelligent question answering!